# MediScan — Day 4: Custom Dataset and DataLoader

## Objective

Create a custom PyTorch Dataset and DataLoaders for the training,
validation, and testing datasets.

The DataLoader pipeline will prepare batches of MRI images for
subsequent model training.

In [1]:
import torch
import pandas as pd

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

print("MyDrive exists:", DRIVE_ROOT.exists())

print("\nFolders inside MyDrive:")
for item in DRIVE_ROOT.iterdir():
    if item.is_dir():
        print("📁", item.name)

MyDrive exists: True

Folders inside MyDrive:
📁 New Folder
📁 OKIE DOKIE APP
📁 Knowledge Base
📁 Nishu Bhabhi
📁 Colab Notebooks
📁 DATA ANALYST INTERNSHIP 
📁 MediScan


In [6]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/MediScan"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "val.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 5040
Validation: 1080
Test: 1080


In [7]:
class BrainMRIDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = row["label"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [9]:
# ============================================
# Day 4 - Transform Pipelines
# ============================================

train_augmentation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Training transform:")
print(train_augmentation)

print("\nEvaluation transform:")
print(eval_transform)

Training transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.85, 1.15), contrast=(0.85, 1.15), saturation=(0.9, 1.1), hue=(-0.02, 0.02))
    RandomAffine(degrees=[0.0, 0.0], translate=(0.05, 0.05), scale=(0.95, 1.05), shear=[-5.0, 5.0])
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Evaluation transform:
Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [10]:
train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=eval_transform
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=eval_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [12]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MediScan")

matches = list(
    PROJECT_DIR.rglob("Tr-no_626.jpg")
)

print("Matches found:", len(matches))

for path in matches:
    print(path)

Matches found: 0


In [13]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

matches = list(
    DRIVE_ROOT.rglob("Tr-no_626.jpg")
)

print("Matches found:", len(matches))

for path in matches[:10]:
    print(path)

Matches found: 0


In [14]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

zip_files = list(DRIVE_ROOT.rglob("archive.zip"))

print("ZIP files found:", len(zip_files))

for path in zip_files:
    print(path)

ZIP files found: 1
/content/drive/MyDrive/archive.zip


In [15]:
import zipfile
from pathlib import Path

ZIP_PATH = Path("/content/drive/MyDrive/archive.zip")
EXTRACT_DIR = Path("/content/drive/MyDrive/MediScan/data/raw")

print("ZIP exists:", ZIP_PATH.exists())

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully!")
print("Extracted to:", EXTRACT_DIR)

ZIP exists: True
Dataset extracted successfully!
Extracted to: /content/drive/MyDrive/MediScan/data/raw


In [16]:
print("Extracted folders:")

for item in EXTRACT_DIR.iterdir():
    print("📁" if item.is_dir() else "📄", item.name)

Extracted folders:
📁 Testing
📁 Training


In [17]:
# ============================================
# Day 4 - Fix Dataset Image Paths
# ============================================

from pathlib import Path
import pandas as pd

RAW_DIR = Path(
    "/content/drive/MyDrive/MediScan/data/raw"
)

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/MediScan/data/processed"
)

def fix_image_path(old_path):
    old_path = Path(old_path)

    # Find the Training/Testing part of the original path
    parts = old_path.parts

    if "Training" in parts:
        idx = parts.index("Training")
        relative_path = Path(*parts[idx:])

    elif "Testing" in parts:
        idx = parts.index("Testing")
        relative_path = Path(*parts[idx:])

    else:
        raise ValueError(
            f"Could not determine dataset split from path: {old_path}"
        )

    return RAW_DIR / relative_path


# Load CSVs
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")


# Fix paths
train_df["image_path"] = train_df["image_path"].apply(fix_image_path)
val_df["image_path"] = val_df["image_path"].apply(fix_image_path)
test_df["image_path"] = test_df["image_path"].apply(fix_image_path)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nExample corrected path:")
print(train_df.iloc[0]["image_path"])

Train: 5040
Validation: 1080
Test: 1080

Example corrected path:
/content/drive/MyDrive/MediScan/data/raw/Training/notumor/Tr-no_626.jpg


In [18]:
# ============================================
# Verify Corrected Image Paths
# ============================================

print("Train path exists:",
      train_df.iloc[0]["image_path"].exists())

print("Validation path exists:",
      val_df.iloc[0]["image_path"].exists())

print("Test path exists:",
      test_df.iloc[0]["image_path"].exists())

print("\nChecking all image paths...")

train_exists = train_df["image_path"].apply(
    lambda x: x.exists()
).all()

val_exists = val_df["image_path"].apply(
    lambda x: x.exists()
).all()

test_exists = test_df["image_path"].apply(
    lambda x: x.exists()
).all()

print("All train images exist:", train_exists)
print("All validation images exist:", val_exists)
print("All test images exist:", test_exists)

Train path exists: True
Validation path exists: True
Test path exists: True

Checking all image paths...
All train images exist: True
All validation images exist: True
All test images exist: True


In [19]:
# ============================================
# Day 4 - Create Custom PyTorch Datasets
# ============================================

train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=eval_transform
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=eval_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [20]:
# ============================================
# Verify One Training Sample
# ============================================

image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Label:", label)
print("Label type:", type(label))
print("Pixel min:", image.min().item())
print("Pixel max:", image.max().item())

Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: notumor
Label type: <class 'str'>
Pixel min: -2.1179039478302
Pixel max: 2.3088455200195312


In [21]:
# ============================================
# Day 4 - Create DataLoaders
# ============================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created successfully!")
print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully!
Batch size: 32
Train batches: 158
Validation batches: 34
Test batches: 34


In [23]:
# ============================================
# Day 4 - Load Existing Class Mapping
# ============================================

import json

CLASS_MAPPING_PATH = (
    PROCESSED_DIR / "class_to_idx.json"
)

with open(CLASS_MAPPING_PATH, "r") as f:
    class_to_idx = json.load(f)

print("Class mapping:")
print(class_to_idx)

Class mapping:
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [24]:
# ============================================
# Day 4 - Updated Custom Dataset
# ============================================

class BrainMRIDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None,
        class_to_idx=None
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_path = row["image_path"]
        label = row["label"]

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Convert class name to integer index
        label = self.class_to_idx[label]

        return image, label

In [25]:
# ============================================
# Recreate Datasets with Integer Labels
# ============================================

train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation,
    class_to_idx=class_to_idx
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=eval_transform,
    class_to_idx=class_to_idx
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=eval_transform,
    class_to_idx=class_to_idx
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [26]:
# ============================================
# Recreate DataLoaders
# ============================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders recreated successfully!")

DataLoaders recreated successfully!


In [27]:
# ============================================
# Final DataLoader Batch Verification
# ============================================

images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

print("\nBatch size:", images.size(0))
print("Image channels:", images.size(1))
print("Image height:", images.size(2))
print("Image width:", images.size(3))

print("\nSample labels:")
print(labels[:10])

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Labels dtype: torch.int64

Batch size: 32
Image channels: 3
Image height: 224
Image width: 224

Sample labels:
tensor([0, 0, 2, 2, 2, 0, 3, 3, 2, 1])


In [28]:
# ============================================
# Day 4 - DataLoader Configuration Verification
# ============================================

print("Train DataLoader")
print("----------------")
print("Batch size :", train_loader.batch_size)
print("Shuffle    :", train_loader.dataset is train_dataset)
print("Workers    :", train_loader.num_workers)
print("Pin memory :", train_loader.pin_memory)

print("\nValidation DataLoader")
print("--------------------")
print("Batch size :", val_loader.batch_size)
print("Workers    :", val_loader.num_workers)
print("Pin memory :", val_loader.pin_memory)

print("\nTest DataLoader")
print("---------------")
print("Batch size :", test_loader.batch_size)
print("Workers    :", test_loader.num_workers)
print("Pin memory :", test_loader.pin_memory)

Train DataLoader
----------------
Batch size : 32
Shuffle    : True
Workers    : 2
Pin memory : True

Validation DataLoader
--------------------
Batch size : 32
Workers    : 2
Pin memory : True

Test DataLoader
---------------
Batch size : 32
Workers    : 2
Pin memory : True


In [29]:
# ============================================
# Day 4 - DataLoader Iteration Test
# ============================================

import time

start_time = time.time()

batch_count = 0

for images, labels in train_loader:
    batch_count += 1

end_time = time.time()

print("DataLoader iteration successful!")
print("Batches loaded:", batch_count)
print(
    "Time taken:",
    round(end_time - start_time, 2),
    "seconds"
)

DataLoader iteration successful!
Batches loaded: 158
Time taken: 45.86 seconds


In [30]:
# ============================================
# Day 4 - Efficient DataLoader with Prefetch
# ============================================

BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

print("Efficient DataLoaders created successfully!")
print("Batch size:", BATCH_SIZE)
print("Workers:", NUM_WORKERS)
print("Prefetch factor:", PREFETCH_FACTOR)
print("Pin memory:", True)

Efficient DataLoaders created successfully!
Batch size: 32
Workers: 2
Prefetch factor: 2
Pin memory: True


In [31]:
# ============================================
# Day 4 - Final DataLoader Verification
# ============================================

images, labels = next(iter(train_loader))

print("========================================")
print("DAY 4 DATALOADER VERIFICATION")
print("========================================")

print("Images shape :", images.shape)
print("Labels shape :", labels.shape)
print("Images dtype :", images.dtype)
print("Labels dtype :", labels.dtype)

print("\nDataLoader configuration:")
print("Batch size   :", train_loader.batch_size)
print("Num workers  :", train_loader.num_workers)
print("Prefetch     :", train_loader.prefetch_factor)
print("Pin memory   :", train_loader.pin_memory)

print("\n========================================")
print("DAY 4 VERIFICATION PASSED")
print("========================================")

DAY 4 DATALOADER VERIFICATION
Images shape : torch.Size([32, 3, 224, 224])
Labels shape : torch.Size([32])
Images dtype : torch.float32
Labels dtype : torch.int64

DataLoader configuration:
Batch size   : 32
Num workers  : 2
Prefetch     : 2
Pin memory   : True

DAY 4 VERIFICATION PASSED
